# KG1 NVIDIA Nemotron - v48 ALL-IN-ONE (PROVEN + DISTILLED)

## Config: 21-agent research consensus + 299 distilled equation CoTs

### Changes from v47e:
- rank 16→**32**, alpha ratio 2:1→**1:1**, lr 3e-5→**1e-4**
- epochs 5→**2**, grad_accum 32→**8**, max_length 1024→**2048**
- warmup 0.15→**0.05**, grad_norm 0.5→**1.0**
- Added **out_proj** to targets (konbu17 proven)
- **299 equation weak CoTs → distilled** (DeepSeek-R1/o4-mini/Gemini)

### Run: Cell 1 → RESTART → Cell 2


In [ ]:
#@title CELL 1: Install (UNIFIED stack: torch 2.8 + cu128 - works on H100 + Blackwell)
#@markdown ### APOS rodar: Runtime > Restart session, depois rode Cell 2
#@markdown ### Versoes PINADAS - sem -U sem upgrade automatico (evita breaking changes)

import subprocess, sys, os

def _sh(cmd):
    """Run shell, print last 1500 chars of stdout/stderr."""
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (r.stdout or "") + (r.stderr or "")
    if out.strip():
        print(out[-1500:])
    return r.returncode

def _pkg_ver(name):
    try:
        import importlib.metadata
        return importlib.metadata.version(name)
    except Exception:
        return None

# ============================================================
# 0. Python version sanity
# ============================================================
py_maj_min = f"{sys.version_info.major}.{sys.version_info.minor}"
py_cpver = f"cp{sys.version_info.major}{sys.version_info.minor}"
print(f"Python: {py_maj_min} ({py_cpver})")
assert sys.version_info >= (3, 10), f"Python {py_maj_min} too old, need 3.10+"
assert sys.version_info < (3, 14), f"Python {py_maj_min} not supported by mamba-ssm wheels"

# ============================================================
# 1. GPU info (informational only - same stack works on all)
# ============================================================
print("\n=== GPU INFO ===")
try:
    r = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv,noheader"],
        capture_output=True, text=True, timeout=10,
    )
    line = r.stdout.strip().split("\n")[0]
    gpu_name, gpu_cap_str = [x.strip() for x in line.split(",", 1)]
    gpu_cap = float(gpu_cap_str)
    print(f"  GPU: {gpu_name}")
    print(f"  Compute: sm_{int(gpu_cap*10)}")
    if gpu_cap >= 10.0:
        print(f"  Architecture: BLACKWELL (sm_{int(gpu_cap*10)}) - needs cu128")
    elif gpu_cap >= 9.0:
        print(f"  Architecture: HOPPER (sm_{int(gpu_cap*10)})")
    elif gpu_cap >= 8.0:
        print(f"  Architecture: AMPERE (sm_{int(gpu_cap*10)})")
    else:
        print(f"  Architecture: older (sm_{int(gpu_cap*10)})")
except Exception as e:
    print(f"  [WARN] nvidia-smi failed: {e}")

# ============================================================
# 2. UNIFIED PROFILE - works on H100 (sm_90) AND Blackwell (sm_120)
# ============================================================
# torch 2.8 + cu128 has kernels for sm_80, sm_86, sm_89, sm_90, sm_100, sm_120
# (CUDA 12.8 is forward-compatible with all NVIDIA architectures Ampere+)

TORCH_PIN = "torch==2.8.0"
TORCH_INDEX = "https://download.pytorch.org/whl/cu128"
TORCH_OK_PREFIX = "2.8."

# mamba-ssm 2.3.1 wheel for torch 2.8 + cu12 + cxx11abiTRUE (matches torch 2.8 default ABI)
MAMBA_REL = "v2.3.1"
MAMBA_FILE = f"mamba_ssm-2.3.1+cu12torch2.8cxx11abiTRUE-{py_cpver}-{py_cpver}-linux_x86_64.whl"
MAMBA_URL = f"https://github.com/state-spaces/mamba/releases/download/{MAMBA_REL}/{MAMBA_FILE}"

# causal-conv1d 1.6.1.post4 wheel for torch 2.8 + cu12 + cxx11abiTRUE
CONV_REL = "v1.6.1.post4"
CONV_FILE = f"causal_conv1d-1.6.1+cu12torch2.8cxx11abiTRUE-{py_cpver}-{py_cpver}-linux_x86_64.whl"
CONV_URL = f"https://github.com/Dao-AILab/causal-conv1d/releases/download/{CONV_REL}/{CONV_FILE}"

# PINNED versions for ML stack (avoids transformers 5.x and trl 1.x breaking changes)
# - transformers 4.57.6: latest 4.x, has GenerateDecoderOnlyOutput (mamba 2.3.1 needs)
# - trl 0.29.1: latest 0.x, has processing_class API + dataset_text_field + max_length
# - peft 0.18.1: latest, supports torch>=1.13
# - accelerate 1.13.0: latest stable
# - bitsandbytes 0.49.2: latest, supports torch>=2.3
ML_DEPS = [
    "transformers==4.57.6",
    "peft==0.18.1",
    "trl==0.29.1",
    "accelerate==1.13.0",
    "bitsandbytes==0.49.2",
    "datasets",
    "safetensors",
    "huggingface_hub",
    "kaggle",
]

# ============================================================
# 3. Check existing install
# ============================================================
print("\n=== CHECKING EXISTING INSTALL ===")
torch_ver = _pkg_ver("torch")
mamba_ver = _pkg_ver("mamba-ssm") or _pkg_ver("mamba_ssm")
conv1d_ver = _pkg_ver("causal-conv1d") or _pkg_ver("causal_conv1d")
print(f"  torch:         {torch_ver}")
print(f"  mamba-ssm:     {mamba_ver}")
print(f"  causal-conv1d: {conv1d_ver}")
print(f"  required torch: {TORCH_OK_PREFIX}*")

torch_ok = torch_ver and torch_ver.startswith(TORCH_OK_PREFIX)
mamba_ok = mamba_ver and mamba_ver.startswith("2.3.1")
conv1d_ok = conv1d_ver and conv1d_ver.startswith("1.6.1")

# CRITICAL: if torch is installed at WRONG version, MUST uninstall everything
if torch_ver and not torch_ok:
    print(f"\n  [REINSTALL] torch {torch_ver} != {TORCH_OK_PREFIX}*")
    print(f"  Removing ALL torch + mamba + conv1d (must rebuild stack)")
    _sh("pip uninstall -y torch torchvision torchaudio triton mamba-ssm mamba_ssm causal-conv1d causal_conv1d")
    torch_ok = False
    mamba_ok = False
    conv1d_ok = False

# Even if torch is right, if mamba/conv1d are wrong version, reinstall them
if mamba_ver and not mamba_ok:
    print(f"\n  [REINSTALL] mamba-ssm {mamba_ver} != 2.3.1")
    _sh("pip uninstall -y mamba-ssm mamba_ssm")
    mamba_ok = False

if conv1d_ver and not conv1d_ok:
    print(f"\n  [REINSTALL] causal-conv1d {conv1d_ver} != 1.6.1")
    _sh("pip uninstall -y causal-conv1d causal_conv1d")
    conv1d_ok = False

NEED_RESTART = False

if torch_ok and mamba_ok and conv1d_ok:
    print("\n  [OK] torch + mamba + conv1d at correct versions")
else:
    NEED_RESTART = True
    print("\n=== INSTALLING ===")

    if not torch_ok:
        print(f"\n  Installing {TORCH_PIN} from {TORCH_INDEX} ...")
        rc = _sh(f"pip install --quiet {TORCH_PIN} --index-url {TORCH_INDEX}")
        if rc != 0:
            raise RuntimeError(f"FATAL: torch install failed (rc={rc}). Check pip output above.")

    if not conv1d_ok:
        print(f"\n  Installing causal-conv1d {CONV_REL} ...")
        print(f"  URL: {CONV_URL}")
        if _sh(f"pip install --quiet {CONV_URL}") != 0:
            raise RuntimeError(f"FATAL: causal-conv1d install failed. URL: {CONV_URL}")

    if not mamba_ok:
        print(f"\n  Installing mamba-ssm {MAMBA_REL} ...")
        print(f"  URL: {MAMBA_URL}")
        if _sh(f"pip install --quiet {MAMBA_URL}") != 0:
            raise RuntimeError(f"FATAL: mamba-ssm install failed. URL: {MAMBA_URL}")

# ============================================================
# 4. Install ML deps (PINNED versions, no -U upgrade)
# ============================================================
print("\n=== INSTALLING ML DEPS (pinned) ===")
deps_str = " ".join(ML_DEPS)
rc = _sh(f"pip install --quiet {deps_str}")
if rc != 0:
    raise RuntimeError(f"FATAL: ML deps install failed (rc={rc})")

# ============================================================
# 5. Final version check
# ============================================================
print("\n=== VERSION CHECK ===")
import importlib

EXPECTED = {
    "torch": "2.8.",
    "mamba_ssm": "2.3.1",
    "causal_conv1d": "1.6.1",
    "transformers": "4.57.",
    "peft": "0.18.",
    "trl": "0.29.",
    "accelerate": "1.13.",
    "bitsandbytes": "0.49.",
}

all_ok = True
for pkg, expected_prefix in EXPECTED.items():
    try:
        m = importlib.import_module(pkg)
        v = getattr(m, "__version__", _pkg_ver(pkg.replace("_", "-")))
        if v and v.startswith(expected_prefix):
            print(f"  [OK]   {pkg}: {v}")
        else:
            print(f"  [FAIL] {pkg}: {v} (expected {expected_prefix}*)")
            all_ok = False
    except Exception as e:
        print(f"  [FAIL] {pkg}: import failed - {e}")
        all_ok = False

print()
print(f"  All critical OK: {all_ok}")

if NEED_RESTART:
    print("\n" + "=" * 60)
    print("  RESTART RUNTIME REQUIRED")
    print("  Click: Runtime > Restart session, then run Cell 2")
    print("=" * 60)
elif not all_ok:
    print("\n  [WARN] Some packages wrong - check above. May need restart anyway.")
else:
    print("\n  [OK] Ready. Run Cell 2.")


In [ ]:
#@title CELL 2: ALL-IN-ONE v48 (PROVEN CONFIG — 21-agent research consensus)
#@markdown ### v48: rank=32, alpha=32, lr=1e-4, 2 epochs — konbu17/atahalam/NVIDIA proven recipe

# ============================================================
# SECTION 0 - IMPORTS + VERSION GUARDS
# ============================================================
import os, sys, json, gc, re, math, time, random, glob, warnings
warnings.filterwarnings("ignore", category=UserWarning)

print("=" * 60)
print("KG1 v48 ALL-IN-ONE - Starting (PROVEN: 21-agent consensus config)")
print("=" * 60)

import torch
import subprocess as _sp
print(f"\n[versions]")
print(f"  torch: {torch.__version__}")
assert torch.cuda.is_available(), "FATAL: CUDA not available"
assert torch.__version__.startswith("2.8."), (
    f"FATAL: torch {torch.__version__} - run Cell 1 (installs torch 2.8.0+cu128) then RESTART"
)

# CRITICAL: test CUDA kernel BEFORE loading 30B model (catches sm mismatch in <1s)
print("  [test] CUDA kernel sanity check ...")
try:
    _t = torch.zeros(64, 64, device="cuda", dtype=torch.bfloat16)
    _ = (_t @ _t).sum().item()
    del _t
    torch.cuda.empty_cache()
    print("  [OK] CUDA kernel works on this GPU")
except RuntimeError as _e:
    raise RuntimeError(
        f"FATAL: CUDA kernel test failed.\nError: {_e}\n"
        f"FIX: Re-run Cell 1 then RESTART runtime."
    ) from _e

# Detect GPU compute capability (Blackwell = sm_120+)
_gpu_props = torch.cuda.get_device_properties(0)
_gpu_cap = float(f"{_gpu_props.major}.{_gpu_props.minor}")
IS_BLACKWELL = _gpu_cap >= 10.0
_gpu_name = torch.cuda.get_device_name(0)
print(f"  GPU: {_gpu_name} (sm_{int(_gpu_cap*10)})")
if IS_BLACKWELL:
    print(f"  [BLACKWELL] sm_{int(_gpu_cap*10)} detected")
    print(f"  [BLACKWELL] Mamba CUDA kernels incompatible - will use naive PyTorch path")

import numpy as np
import pandas as pd
from collections import Counter

import transformers, peft, trl, datasets
print(f"  transformers: {transformers.__version__}")
print(f"  peft: {peft.__version__}")
print(f"  trl: {trl.__version__}")
print(f"  datasets: {datasets.__version__}")

# Try import mamba (optional - will use naive path if missing)
try:
    import mamba_ssm
    print(f"  mamba_ssm: {getattr(mamba_ssm, '__version__', 'unknown')} [FAST PATH]")
except Exception:
    print(f"  mamba_ssm: NOT AVAILABLE [NAIVE PATH - 3-5x slower]")

try:
    import causal_conv1d
    print(f"  causal_conv1d: {getattr(causal_conv1d, '__version__', 'unknown')} [FAST]")
except Exception:
    print(f"  causal_conv1d: NOT AVAILABLE")

from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
    TrainerCallback,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
from huggingface_hub import HfApi, hf_hub_download, create_repo, login

# ============================================================
# SECTION 1 - CONFIG (v48: MoE-stable)
# ============================================================
CFG = {
    # Model
    "model_name": "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16",
    "data_repo": "felipesp1983/kg1-nemotron-training",
    "run_tag": "v48-proven-consensus",

    # LoRA — v48: PROVEN config (100% consensus from 9 top notebooks + 21 agents)
    # ALL proven notebooks (konbu17 328v, atahalam 151v, kienngx 602v, kai-xu) use rank=32
    "lora_rank": 32,                  # v48: 16->32 (UNANIMOUS consensus, max_lora_rank limit)
    "lora_alpha": 32,                 # v48: ratio 1:1 (konbu17 + kai-xu proven)
    "lora_dropout": 0.05,
    "use_dora": False,                # KEEP: PEFT #2274 fix (Mamba-2 grad)
    "target_modules": [
        "in_proj",                    # Mamba-2 input (konbu17 proven)
        "out_proj",                   # Mamba-2 output (konbu17 proven, was MISSING in v48!)
        "q_proj", "k_proj", "v_proj", "o_proj",  # Attention (v30 proven)
        "up_proj", "down_proj",       # MLP / shared_experts (konbu17 proven)
    ],
    "exclude_modules": r".*\.experts\.\d+\..*",  # KEEP: exclude routed experts (v46 root cause fix)

    # Training — v48: konbu17/atahalam PROVEN hyperparams
    "learning_rate": 1e-4,            # v48: 3e-5->1e-4 (konbu17/atahalam/mayukh ALL use 1e-4)
    "n_epochs": 2,                    # v48: 5->2 (konbu17/v30 consensus, 5 = overfitting)
    "grad_accum": 8,                  # v48: 32->8 (konbu17 uses 8, more noise = better generalization)
    "warmup_ratio": 0.05,             # v48: 0.15->0.05 (konbu17/v30 consensus)
    "max_length": 2048,               # v48: 1024->2048 (konbu17 uses 4096, 2048 = safe minimum for CoT)
    "max_grad_norm": 1.0,             # v48: 0.5->1.0 (v30/default proven)
    "seed": 123,

    # Auto-submit schedule (MUST be multiples of save_steps=100)
    # v48: 2 epochs × 5858/8 ≈ 1465 steps → submit early + mid + late
    "submit_steps": [100, 200, 400, 700, 1000, 1400],

    # Safety (thresholds are for NORMALIZED loss = raw / grad_accum)
    "freeze_moe_router": True,
    "smoke_min_loss": 0.3,
    "smoke_max_loss": 10.0,
    "smoke_max_steps": 10,            # v48: 10 steps sufficient (grad_accum=8 = faster steps)
    "smoke_n_examples": 128,          # 128/8=16 epochs in smoke → good coverage
    "smoke_max_first_ratio": 2.5,
    "smoke_last_first_ratio": 1.5,
}

OUTPUT_DIR = "/content/lora_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"\n[config]")
for k, v in CFG.items():
    print(f"  {k}: {v}")

# Reproducibility
random.seed(CFG["seed"])
np.random.seed(CFG["seed"])
torch.manual_seed(CFG["seed"])
torch.cuda.manual_seed_all(CFG["seed"])

# ============================================================
# SECTION 2 - AUTH (HF_KEY + KAGGLE)
# ============================================================
print("\n=== AUTH ===")

def _get_secret(*names):
    """Try Colab userdata first, then env vars."""
    try:
        from google.colab import userdata
        for n in names:
            v = userdata.get(n)
            if v:
                return v
    except Exception:
        pass
    for n in names:
        v = os.environ.get(n)
        if v:
            return v
    return None

HF_TOKEN = _get_secret("HF_KEY", "HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("FATAL: HF_KEY not in Colab Secrets (sidebar > Secrets > Add new secret)")
login(token=HF_TOKEN, add_to_git_credential=False)
os.environ["HF_TOKEN"] = HF_TOKEN
print(f"  [OK] HF login")

K_USER = _get_secret("KAGGLE_USERNAME")
K_KEY = _get_secret("KAGGLE_KEY")
KAGGLE_READY = False
if K_USER and K_KEY:
    os.environ["KAGGLE_USERNAME"] = K_USER
    os.environ["KAGGLE_KEY"] = K_KEY
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
        json.dump({"username": K_USER, "key": K_KEY}, f)
    os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    KAGGLE_READY = True
    print(f"  [OK] Kaggle: {K_USER}")
else:
    print(f"  [WARN] Kaggle secrets missing - manual submit only (HF upload still works)")

# ============================================================
# SECTION 3 - DOWNLOAD DATA
# ============================================================
print("\n=== DOWNLOAD DATA ===")
# v48: Use MERGED data (solver + distilled CoTs) — 299 weak→distilled upgrades
_data_file = "data/solver_augmented_train_v48.jsonl"
try:
    hf_hub_download(
        repo_id=CFG["data_repo"],
        filename=_data_file,
        local_dir="/tmp/kg1_data",
        repo_type="dataset",
    )
    SOLVER_JSONL = f"/tmp/kg1_data/{_data_file}"
    print(f"  [OK] Using v48 merged data (distilled CoTs)")
except Exception as _e:
    print(f"  [WARN] v48 data not found ({_e}), falling back to original")
    hf_hub_download(
        repo_id=CFG["data_repo"],
        filename="data/solver_augmented_train.jsonl",
        local_dir="/tmp/kg1_data",
        repo_type="dataset",
    )
    SOLVER_JSONL = "/tmp/kg1_data/data/solver_augmented_train.jsonl"

raw_rows = []
with open(SOLVER_JSONL, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            raw_rows.append(json.loads(line))
print(f"  [OK] loaded {len(raw_rows)} rows")
assert len(raw_rows) >= 6000, f"FATAL: only {len(raw_rows)} rows, expected >=6000"

# Filter answer length <= 24 (Kaggle DATA_GATE_POLICY)
filtered_rows = [r for r in raw_rows if len(str(r.get("answer", ""))) <= 24]
print(f"  After ans_len<=24: {len(filtered_rows)}")

FAMILY_MAP = {
    "gravity_constant": "grav",
    "unit_conversion": "unit",
    "numeral_system": "num",
    "text_encryption": "enc",
    "bit_manipulation": "bit",
    "equation_transform": "eq",
}
for r in filtered_rows:
    r["fam"] = FAMILY_MAP.get(r.get("family", ""), "other")

fam_counts = Counter(r["fam"] for r in filtered_rows)
print(f"  Family distribution:")
for fam in sorted(fam_counts):
    print(f"    {fam:6s}: {fam_counts[fam]:4d}")

random.shuffle(filtered_rows)

# ============================================================
# SECTION 4 - TOKENIZER + PAD TOKEN FIX
# ============================================================
print("\n=== TOKENIZER ===")
tokenizer = AutoTokenizer.from_pretrained(CFG["model_name"], trust_remote_code=True)

print(f"  Before: pad={tokenizer.pad_token!r}(id={tokenizer.pad_token_id}), eos={tokenizer.eos_token!r}(id={tokenizer.eos_token_id})")

# CRITICAL: pad_token must differ from eos_token (else EOS gets -100 masked)
if tokenizer.pad_token is None or tokenizer.pad_token_id == tokenizer.eos_token_id:
    vocab = tokenizer.get_vocab()
    candidates = ["<|pad|>", "<|extra_0|>", "<|reserved_0|>", "<|unused_0|>", "<unk>"]
    picked = None
    for c in candidates:
        if c in vocab and vocab[c] != tokenizer.eos_token_id:
            picked = c
            break
    if picked:
        tokenizer.pad_token = picked
        print(f"  Using reserved token as pad: {picked!r}")
    else:
        tokenizer.add_special_tokens({"pad_token": "<|PAD|>"})
        print(f"  Added new pad_token <|PAD|> (will resize embeddings)")

print(f"  After:  pad={tokenizer.pad_token!r}(id={tokenizer.pad_token_id}), eos={tokenizer.eos_token!r}(id={tokenizer.eos_token_id})")
assert tokenizer.pad_token_id != tokenizer.eos_token_id, "FATAL: pad_token_id must differ from eos_token_id"

# ============================================================
# SECTION 5 - BUILD DATASET
# ============================================================
print("\n=== BUILD DATASET ===")
PROMPT_SUFFIX = "\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`"

def extract_reasoning(content, fallback):
    """Strip trailing \\boxed{...} from assistant content."""
    m = re.search(r"\\boxed\{([^}]*)\}\s*$", content)
    if m:
        return content[:m.start()].rstrip(), m.group(1)
    return content.rstrip(), fallback

examples = []
skipped = 0
for row in filtered_rows:
    prompt = str(row["prompt"])
    row_answer = str(row["answer"])
    msgs = row.get("messages", [])
    asst_msg = next((m for m in msgs if m.get("role") == "assistant"), None)
    if not asst_msg:
        skipped += 1
        continue
    orig = asst_msg.get("content", "")
    if not orig.strip():
        skipped += 1
        continue
    reasoning, _ = extract_reasoning(orig, row_answer)
    if len(reasoning.strip()) < 5:
        reasoning = orig.rstrip()

    asst_content = f"<think>\n{reasoning}\n</think>\n\\boxed{{{row_answer}}}"
    user_content = prompt + PROMPT_SUFFIX
    examples.append({
        "messages": [
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": asst_content},
        ]
    })

print(f"  Built {len(examples)} examples (skipped {skipped})")
assert len(examples) >= 5000, f"FATAL: only {len(examples)} examples, expected >=5000"

# Apply chat template -> single text field
texts = []
for ex in examples:
    text = tokenizer.apply_chat_template(
        ex["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    texts.append(text)

ds = Dataset.from_dict({"text": texts})
print(f"  Dataset: {len(ds)} examples")

# Token length stats
lens = [len(tokenizer.encode(t)) for t in texts[:300]]
print(f"  Token lengths (first 300): avg={np.mean(lens):.0f}, max={max(lens)}, p95={np.percentile(lens,95):.0f}, p99={np.percentile(lens,99):.0f}")
if max(lens) > CFG["max_length"]:
    n_trunc = sum(1 for l in lens if l > CFG["max_length"])
    print(f"  [WARN] {n_trunc}/300 samples exceed max_length={CFG['max_length']}")

print(f"\n  Sample[0] (first 600 chars):\n{texts[0][:600]}")

# ============================================================
# SECTION 6 - GPU MODE SELECTION (NF4 vs BF16 full)
# ============================================================
print("\n=== GPU MODE ===")
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
gpu_name = torch.cuda.get_device_name(0)
print(f"  GPU: {gpu_name}")
print(f"  VRAM: {gpu_mem_gb:.1f} GB")

USE_NF4 = gpu_mem_gb < 70
_mode_label = "NF4 4-bit (low VRAM fallback)" if USE_NF4 else "BF16 full precision"
print(f"  Mode: {_mode_label}")

# ============================================================
# SECTION 7 - LOAD MODEL (with Blackwell mamba kernel override)
# ============================================================
print("\n=== LOAD MODEL (takes 5-8 min) ===")

# Load config first to override Mamba kernels for Blackwell
from transformers import AutoConfig
model_config = AutoConfig.from_pretrained(CFG["model_name"], trust_remote_code=True)

# CRITICAL: mamba_ssm 2.3.1 CUDA kernels produce GARBAGE on Blackwell sm_120
# (v48 evidence: step 1 loss=65.5 with mamba_kernels=True vs 1.6 on H100)
# Naive PyTorch path is ~2-3x slower but produces CORRECT results on any GPU
if IS_BLACKWELL and hasattr(model_config, "use_mamba_kernels"):
    model_config.use_mamba_kernels = False
    print(f"  [BLACKWELL] use_mamba_kernels=False (naive PyTorch path)")
    print(f"  [BLACKWELL] Training ~2-3x slower but CORRECT on sm_{int(_gpu_cap*10)}")
elif hasattr(model_config, "use_mamba_kernels"):
    print(f"  [OK] use_mamba_kernels=True (fast CUDA/Triton path on sm_{int(_gpu_cap*10)})")

if USE_NF4:
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        CFG["model_name"],
        config=model_config,
        quantization_config=bnb,
        device_map={"": 0},
        trust_remote_code=True,
        torch_dtype=torch.bfloat16,
    )
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
else:
    model = AutoModelForCausalLM.from_pretrained(
        CFG["model_name"],
        config=model_config,
        device_map={"": 0},
        trust_remote_code=True,
        torch_dtype=torch.bfloat16,
    )
    model.gradient_checkpointing_enable()

print(f"  [OK] model loaded. dtype={next(model.parameters()).dtype}")

emb_size = model.get_input_embeddings().weight.shape[0]
if len(tokenizer) > emb_size:
    print(f"  Resizing embeddings: {emb_size} -> {len(tokenizer)}")
    model.resize_token_embeddings(len(tokenizer))

# ============================================================
# SECTION 8 - MoE CLASS-LEVEL PATCH (dtype fix) + STRUCTURE AUDIT
# ============================================================
print("\n=== MoE PATCH ===")
_patched = []
for mod_name, mod in list(sys.modules.items()):
    if "nemotron_h" not in mod_name.lower():
        continue
    for attr_name in dir(mod):
        try:
            cls = getattr(mod, attr_name, None)
        except Exception:
            continue
        if not isinstance(cls, type):
            continue
        name_upper = attr_name.upper()
        if "MOE" in name_upper and hasattr(cls, "forward"):
            orig = cls.forward
            def _patched_forward(self, hidden_states, *args, _orig=orig, **kwargs):
                out = _orig(self, hidden_states, *args, **kwargs)
                if isinstance(out, tuple):
                    return tuple(
                        x.to(hidden_states.dtype) if isinstance(x, torch.Tensor) else x
                        for x in out
                    )
                return out.to(hidden_states.dtype) if isinstance(out, torch.Tensor) else out
            cls.forward = _patched_forward
            _patched.append(f"{mod_name}.{attr_name}")

print(f"  Patched MoE classes: {_patched if _patched else 'NONE'}")

# === MODEL STRUCTURE AUDIT (v48) ===
print("\n=== MODEL STRUCTURE AUDIT ===")
n_routed_expert_blocks = 0
n_shared_expert_blocks = 0
n_attn_blocks = 0
n_mamba_blocks = 0
n_mlp_blocks = 0
n_router_params = 0

for n, m in model.named_modules():
    if re.search(r"\.experts\.\d+$", n):
        n_routed_expert_blocks += 1
    elif n.endswith(".shared_experts"):
        n_shared_expert_blocks += 1

for n, p in model.named_parameters():
    if n.endswith(".gate.weight") and ".mixer.gate.weight" in n:
        n_router_params += 1

# Count by mixer type via class name
for n, m in model.named_modules():
    if not n.endswith(".mixer"):
        continue
    cls_name = type(m).__name__
    if "Mamba" in cls_name:
        n_mamba_blocks += 1
    elif "Attention" in cls_name or "Attn" in cls_name:
        n_attn_blocks += 1
    elif "MOE" in cls_name.upper() or "MoE" in cls_name:
        pass  # counted via experts
    elif "MLP" in cls_name.upper():
        n_mlp_blocks += 1

n_moe_blocks = n_shared_expert_blocks  # 1 shared per MoE
print(f"  Mamba blocks (mixer): {n_mamba_blocks}")
print(f"  Attention blocks (mixer): {n_attn_blocks}")
print(f"  MLP blocks (mixer, non-MoE): {n_mlp_blocks}")
print(f"  MoE blocks (mixer, w/ shared_experts): {n_moe_blocks}")
print(f"  Routed experts total: {n_routed_expert_blocks}")
print(f"  Router (gate.weight) params: {n_router_params}")

# Verify router will be auto-frozen by peft (NOT in target_modules)
assert n_router_params >= 0, "Router enumeration failed"
print(f"  [INFO] Router params kept FROZEN (not in target_modules) - auto-handled by peft")

# ============================================================
# SECTION 9 - APPLY LoRA (v48: exclude routed experts)
# ============================================================
print("\n=== APPLY LoRA ===")
print(f"  target_modules: {CFG['target_modules']}")
print(f"  exclude_modules: {CFG['exclude_modules']}")

lora_config = LoraConfig(
    r=CFG["lora_rank"],
    lora_alpha=CFG["lora_alpha"],
    lora_dropout=CFG["lora_dropout"],
    target_modules=CFG["target_modules"],
    exclude_modules=CFG["exclude_modules"],  # NEW v48: skip 128 routed experts
    bias="none",
    task_type="CAUSAL_LM",
    use_dora=CFG["use_dora"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Enable grads on LoRA explicitly
for n, p in model.named_parameters():
    if "lora_" in n:
        p.requires_grad = True

# === CRITICAL VERIFICATION (v48): assert no routed experts got LoRA ===
_routed_re = re.compile(r"\.experts\.\d+\.")
trainable_names = [n for n, p in model.named_parameters() if p.requires_grad]
lora_on_routed = [n for n in trainable_names if _routed_re.search(n)]

print(f"\n  Trainable param tensors: {len(trainable_names)}")
print(f"  LoRA on ROUTED experts: {len(lora_on_routed)} (MUST be 0)")

assert len(lora_on_routed) == 0, (
    f"\nFATAL: {len(lora_on_routed)} routed expert LoRA params found - exclude_modules failed!\n"
    f"First 5: {lora_on_routed[:5]}\n"
    f"This is the v46 explosion root cause - aborting before training."
)
print(f"  [OK] No LoRA on routed experts - v46 root cause ELIMINATED")

# Verify router is frozen
unfrozen_routers = [
    n for n, p in model.named_parameters()
    if n.endswith(".gate.weight") and ".mixer.gate.weight" in n and p.requires_grad
]
assert len(unfrozen_routers) == 0, f"FATAL: {len(unfrozen_routers)} routers unfrozen: {unfrozen_routers[:3]}"
print(f"  [OK] All MoE routers frozen ({n_router_params} gate.weight params)")

# Sample LoRA module names (visual sanity)
lora_module_paths = sorted(set(
    re.sub(r"\.lora_[AB]\.default\.weight$", "", n)
    for n in trainable_names if "lora_" in n
))
print(f"  Unique LoRA modules: {len(lora_module_paths)}")
print(f"  Sample LoRA modules (first 6, last 3):")
for m in lora_module_paths[:6]:
    print(f"    {m}")
if len(lora_module_paths) > 9:
    print(f"    ... ({len(lora_module_paths)-9} more) ...")
for m in lora_module_paths[-3:]:
    print(f"    {m}")

# ============================================================
# SECTION 10 - SMOKE TEST (v48: 20 steps + grad_accum=32 = real config)
# ============================================================
print(f"\n=== SMOKE TEST ({CFG['smoke_max_steps']} steps + grad_accum={CFG['grad_accum']}) ===")
print(f"  This mirrors REAL training config to catch divergence BEFORE 4h waste")

smoke_n = min(CFG["smoke_n_examples"], len(ds))
smoke_args = SFTConfig(
    output_dir="/tmp/smoke_out",
    num_train_epochs=10,  # high enough that max_steps controls
    max_steps=CFG["smoke_max_steps"],
    per_device_train_batch_size=1,
    gradient_accumulation_steps=CFG["grad_accum"],  # SAME as real training
    learning_rate=CFG["learning_rate"],
    warmup_ratio=CFG["warmup_ratio"],
    lr_scheduler_type="cosine",
    logging_steps=1,
    save_strategy="no",
    report_to="none",
    bf16=True,
    gradient_checkpointing=True,
    remove_unused_columns=False,
    dataset_text_field="text",
    max_length=CFG["max_length"],
    max_grad_norm=CFG["max_grad_norm"],
    seed=CFG["seed"],
    optim="adamw_torch",
    dataloader_num_workers=0,
)

smoke_trainer = SFTTrainer(
    model=model,
    train_dataset=ds.select(range(smoke_n)),
    processing_class=tokenizer,
    args=smoke_args,
)
smoke_trainer.train()

smoke_losses_raw = [
    float(l["loss"]) for l in smoke_trainer.state.log_history
    if "loss" in l and l["loss"] is not None
]

# CRITICAL FIX v48: HF Trainer 4.57.6 + Accelerate reports ACCUMULATED loss
# (sum over grad_accum micro-batches), NOT per-sample average.
# Normalize to per-sample for human-readable thresholds.
_ga = CFG["grad_accum"]
smoke_losses = [x / _ga for x in smoke_losses_raw]
print(f"\n  Smoke losses ({len(smoke_losses)} steps, normalized ÷{_ga}):")
for i, (raw, norm) in enumerate(zip(smoke_losses_raw, smoke_losses), 1):
    print(f"    step {i:2d}: {norm:.4f}  (raw: {raw:.1f})")

assert len(smoke_losses) >= 5, f"SMOKE FAIL: only {len(smoke_losses)} loss logs"

# FIX v48: Filter out epoch-boundary steps (partial batches have much lower raw loss).
# With N examples and grad_accum=G, every ceil(N/G) steps has a partial batch.
# These steps have raw loss proportional to (N%G)/G of a normal step.
# Detect: any step with raw loss < 50% of median raw loss is a partial-batch step.
_median_raw = sorted(smoke_losses_raw)[len(smoke_losses_raw) // 2]
_full_step_indices = [i for i, r in enumerate(smoke_losses_raw) if r >= _median_raw * 0.5]
_partial_count = len(smoke_losses_raw) - len(_full_step_indices)
if _partial_count > 0:
    print(f"  [INFO] Filtered out {_partial_count} epoch-boundary steps (partial batches)")
smoke_full = [smoke_losses[i] for i in _full_step_indices]

assert len(smoke_full) >= 3, f"SMOKE FAIL: only {len(smoke_full)} full-batch loss logs"
first_loss = smoke_full[0]
last_loss = smoke_full[-1]
max_loss = max(smoke_full)

# Sanity checks on FULL-BATCH steps only (NaN/Inf, masking, explosion)
assert not any(math.isnan(x) or math.isinf(x) for x in smoke_full), "SMOKE FAIL: NaN/Inf in losses"
assert all(x >= CFG["smoke_min_loss"] for x in smoke_full), (
    f"SMOKE FAIL: loss below {CFG['smoke_min_loss']} (masking bug)"
)
assert max_loss <= CFG["smoke_max_loss"], (
    f"SMOKE FAIL: max loss {max_loss:.2f} > {CFG['smoke_max_loss']} (instability)"
)

# Trajectory checks on full-batch steps (catches MoE divergence)
assert max_loss < first_loss * CFG["smoke_max_first_ratio"], (
    f"SMOKE FAIL: max_loss {max_loss:.2f} > first_loss {first_loss:.2f} * {CFG['smoke_max_first_ratio']}\n"
    f"  -> Indicates divergence trend (MoE instability or bad LR)"
)
assert last_loss < first_loss * CFG["smoke_last_first_ratio"], (
    f"SMOKE FAIL: last_loss {last_loss:.2f} > first_loss {first_loss:.2f} * {CFG['smoke_last_first_ratio']}\n"
    f"  -> Loss not converging (model not learning or LR too high)"
)

print(f"\n  [OK] SMOKE PASS:")
print(f"    first={first_loss:.4f}  last={last_loss:.4f}  max={max_loss:.4f}")
print(f"    last/first ratio: {last_loss/first_loss:.2f} (must be < {CFG['smoke_last_first_ratio']})")
print(f"    max/first ratio:  {max_loss/first_loss:.2f} (must be < {CFG['smoke_max_first_ratio']})")

# Cleanup smoke trainer
del smoke_trainer
gc.collect()
torch.cuda.empty_cache()

# FIX v48 CRITICAL: Reset LoRA weights to fresh init
# Smoke test ran 20 optimizer steps, modifying lora_B from zeros.
# Real training must start from clean LoRA (B=0 → net zero contribution).
print("  Resetting LoRA weights to fresh init (smoke modified them)...")
_reset_count = 0
with torch.no_grad():
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if "lora_B" in name:
            param.zero_()
            _reset_count += 1
        # lora_A keeps its random init — B=0 means LoRA output = A×B = 0 regardless
print(f"  [OK] Reset {_reset_count} lora_B matrices to zero (clean start for real training)")

# ============================================================
# SECTION 11 - REAL TRAINING with CALLBACK
# ============================================================
print("\n=== REAL TRAINING ===")

HF_REPO_ID = f"felipesp1983/kg1-nemotron-lora-{CFG['run_tag']}"
hf_api = HfApi(token=HF_TOKEN)

class GuardedSubmitCallback(TrainerCallback):
    def __init__(self, submit_steps, output_dir, hf_repo_id, api, hf_token):
        self.submit_steps = set(submit_steps)
        self.output_dir = output_dir
        self.hf_repo_id = hf_repo_id
        self.api = api
        self.hf_token = hf_token
        self.uploaded = set()
        self.first_loss = None

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        raw_loss = logs.get("loss")
        step = state.global_step
        if not isinstance(raw_loss, (int, float)):
            return

        # CRITICAL FIX v48: normalize loss (HF Trainer reports accumulated sum)
        loss = raw_loss / args.gradient_accumulation_steps

        if self.first_loss is None and step >= 1:
            self.first_loss = loss

        # GUARD 1: NaN/Inf
        if math.isnan(loss) or math.isinf(loss):
            print(f"\n!!! STOP: NaN/Inf loss at step {step}")
            control.should_training_stop = True
            return

        # GUARD 2: loss=0 (masking/DoRA bug)
        if loss < CFG["smoke_min_loss"] and step > 2:
            print(f"\n!!! STOP: loss={loss:.4f} < {CFG['smoke_min_loss']} at step {step} (masking bug)")
            control.should_training_stop = True
            return

        # GUARD 3: explosion (normalized > 10 = raw > 320)
        # FIX v48: was 30.0 (= raw 960) - way too generous, never triggered
        if loss > 10.0 and step > 5:
            print(f"\n!!! STOP: loss={loss:.4f} > 10 at step {step} (explosion)")
            control.should_training_stop = True
            return

        # GUARD 4: relative explosion (loss > first*3 after step 20)
        if self.first_loss is not None and step > 20 and loss > self.first_loss * 3:
            print(f"\n!!! STOP: loss={loss:.2f} > first_loss={self.first_loss:.2f}*3 at step {step}")
            control.should_training_stop = True
            return

        # GUARD 5: step 10 calibration print (FIX v48: calibrated for normalized scale)
        if step == 10:
            tag = "healthy" if loss < 4.0 else ("high" if loss < 7.0 else "VERY HIGH")
            fl = f"  first={self.first_loss:.4f}" if self.first_loss else ""
            print(f"\n[step10] loss={loss:.4f} [{tag}]{fl}")

    def on_save(self, args, state, control, **kwargs):
        step = state.global_step
        if step not in self.submit_steps or step in self.uploaded:
            return
        self.uploaded.add(step)
        ckpt_dir = os.path.join(self.output_dir, f"checkpoint-{step}")
        if not os.path.isdir(ckpt_dir):
            print(f"\n[WARN] checkpoint {ckpt_dir} not found")
            return
        repo_id = f"{self.hf_repo_id}-step{step}"
        print(f"\n=== AUTO-UPLOAD checkpoint-{step} -> {repo_id} ===")
        try:
            create_repo(repo_id, token=self.hf_token, repo_type="model", exist_ok=True, private=True)
            self.api.upload_folder(
                folder_path=ckpt_dir,
                repo_id=repo_id,
                repo_type="model",
                token=self.hf_token,
            )
            print(f"  [OK] uploaded to {repo_id}")
        except Exception as e:
            print(f"  [ERROR] upload failed: {e}")

train_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=CFG["n_epochs"],
    per_device_train_batch_size=1,
    gradient_accumulation_steps=CFG["grad_accum"],
    learning_rate=CFG["learning_rate"],
    warmup_ratio=CFG["warmup_ratio"],
    lr_scheduler_type="cosine",
    logging_steps=10,
    logging_first_step=True,          # FIX v48: see first step loss immediately
    save_strategy="steps",
    save_steps=100,
    save_total_limit=10,
    report_to="none",
    bf16=True,
    gradient_checkpointing=True,
    remove_unused_columns=False,
    dataset_text_field="text",
    max_length=CFG["max_length"],
    max_grad_norm=CFG["max_grad_norm"],
    weight_decay=0.01,                # Regularization (proven across notebooks)
    seed=CFG["seed"],
    optim="adamw_torch",
    dataloader_num_workers=0,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=ds,
    processing_class=tokenizer,
    args=train_args,
    callbacks=[GuardedSubmitCallback(
        submit_steps=CFG["submit_steps"],
        output_dir=OUTPUT_DIR,
        hf_repo_id=HF_REPO_ID,
        api=hf_api,
        hf_token=HF_TOKEN,
    )],
)

total_steps = (len(ds) * CFG["n_epochs"]) // CFG["grad_accum"]
print(f"\n  Dataset: {len(ds)}")
print(f"  Epochs: {CFG['n_epochs']}")
print(f"  Total optim steps: ~{total_steps}")
print(f"  Submit at: {CFG['submit_steps']}")
print(f"  LR: {CFG['learning_rate']}, warmup: {CFG['warmup_ratio']}, clip: {CFG['max_grad_norm']}")
print(f"  LoRA: rank={CFG['lora_rank']}, alpha={CFG['lora_alpha']}, DoRA={CFG['use_dora']}")
print(f"\n  Starting training...")

trainer.train()

# ============================================================
# SECTION 12 - SAVE + UPLOAD FINAL
# ============================================================
print("\n=== SAVE FINAL ===")
final_dir = os.path.join(OUTPUT_DIR, "final")
trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)
print(f"  [OK] saved to {final_dir}")

try:
    create_repo(HF_REPO_ID, token=HF_TOKEN, repo_type="model", exist_ok=True, private=True)
    hf_api.upload_folder(
        folder_path=final_dir,
        repo_id=HF_REPO_ID,
        repo_type="model",
        token=HF_TOKEN,
    )
    print(f"  [OK] uploaded final to {HF_REPO_ID}")
except Exception as e:
    print(f"  [ERROR] upload final: {e}")

print("\n" + "=" * 60)
print("CELL 2 v48 COMPLETE — DEFINITIVE (6-agent audit)")
print("=" * 60)
print(f"  Checkpoints: {OUTPUT_DIR}")
print(f"  HF repo:     {HF_REPO_ID}")
print(f"  Next: strip_moe_experts + Kaggle submit")
